# Lab 1: Pipeline Integrity

**Scenario:** Ravenwood Recruiting Company needs a defensible Monday planning view.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TheDeafOne/USARD.git"
REPO_DIR = Path("/content/USARD")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already cloned.")

%cd {REPO_DIR}


## Mission

A commander asks: Can we trust the Friday data pull enough to prioritize markets and recruiter attention next week?

You will standardize stages, flag unreliable records, reconcile operational sources, and produce a decision-readiness handoff for the recommender lab. The goal is to make uncertainty visible and prevent unreliable records from influencing later analysis.


## R.O.A.D. framing

- Requirements: Determine whether the weekly pipeline is reliable enough for market-level planning.
- Operationalize data: Establish definitions, source ownership, validation rules, and a handoff.
- Analytics: Later labs rank markets, not individual people.
- Deployment: A leader reviews warnings and approves any plan.

Work in small increments: build one function, run its checks, inspect the output, and only then continue.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 20)
SCENARIO_DATE = pd.Timestamp('2026-08-14')
root_candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((path for path in root_candidates if (path / 'data' / 'pipeline_integrity').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the lab repository or its notebooks directory.')
DATA_DIR = ROOT / 'data' / 'pipeline_integrity'
ARTIFACT_DIR = ROOT / 'artifacts' / 'pipeline_integrity'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Using data from: {DATA_DIR}')


In [ ]:
pipeline = pd.read_csv(DATA_DIR / 'crm_pipeline_extract.csv', parse_dates=['last_stage_at', 'extract_date'])
activity = pd.read_csv(DATA_DIR / 'recruiter_activity_extract.csv', parse_dates=['occurred_at'])
campaign = pd.read_csv(DATA_DIR / 'campaign_lead_extract.csv', parse_dates=['captured_at'])
stage_reference = pd.read_csv(DATA_DIR / 'stage_reference.csv')
capacity = pd.read_csv(DATA_DIR / 'recruiter_capacity_snapshot.csv', parse_dates=['as_of_date'])
labels = pd.read_csv(DATA_DIR / 'stalled_pipeline_labeling.csv')

sources = {'CRM pipeline': pipeline, 'Recruiter activity': activity, 'Campaign leads': campaign, 'Stage reference': stage_reference, 'Recruiter capacity': capacity, 'Stalled-pipeline labels': labels}
pd.DataFrame([{'source': name, 'rows': len(frame), 'columns': ', '.join(frame.columns)} for name, frame in sources.items()])


## Baseline inspection

Inspect what arrived before writing validation logic. Which source owns pipeline stage? Which sources provide supporting evidence? Which snapshot might be too stale for a Friday planning cycle?


In [ ]:
def source_profile(name, frame):
    return {'source': name, 'rows': len(frame), 'duplicate_rows': int(frame.duplicated().sum()), 'missing_values': int(frame.isna().sum().sum())}

display(pd.DataFrame([source_profile(name, frame) for name, frame in sources.items()]))
display(pipeline)
display(stage_reference)


## Task 1: Standardize pipeline stages

The CRM export contains several names for the same operational stage. Use the stage reference to map raw stage to canonical stage and stage rank. Do not invent a mapping for In Review; leave it unmapped so the validation workflow can surface it.

Done when: every recognized stage has a canonical stage and rank, In Review remains unmapped, and raw columns are preserved for traceability.


In [ ]:
def standardize_stages(pipeline, stage_reference):
    '''Return a copy of pipeline with canonical_stage and stage_rank columns.'''
    # TODO: Create lookups from raw_stage to canonical_stage and stage_rank.
    # TODO: Map the lookups onto a copy of pipeline.
    # TODO: Return the standardized copy without dropping raw columns.
    raise NotImplementedError('Complete Task 1 before running this cell.')


In [ ]:
pipeline_standardized = standardize_stages(pipeline, stage_reference)
assert len(pipeline_standardized) == len(pipeline)
assert pipeline_standardized.loc[pipeline_standardized['prospect_id'].eq('P-1006'), 'canonical_stage'].notna().all()
assert pipeline_standardized.loc[pipeline_standardized['prospect_id'].eq('P-1009'), 'canonical_stage'].isna().all()
assert pipeline_standardized.loc[pipeline_standardized['prospect_id'].eq('P-1003'), 'stage_rank'].iloc[0] == 3
display(pipeline_standardized[['prospect_id', 'raw_stage', 'canonical_stage', 'stage_rank']])
print('Task 1 checks passed.')


## Task 2: Validate the CRM pipeline

Create one issue record per affected prospect and rule. Return source, prospect_id, rule, severity, and detail. Flag problems; do not silently repair them.

| Rule | Severity | Condition |
|---|---|---|
| duplicate_prospect_id | MEDIUM | More than one CRM row has the same prospect ID |
| missing_stage_date | HIGH | last_stage_at is missing |
| future_stage_date | HIGH | last_stage_at is after the scenario date |
| unmapped_stage | HIGH | The raw stage is absent from the reference mapping |
| invalid_zip | MEDIUM | ZIP code is not exactly five digits |


In [ ]:
ISSUE_COLUMNS = ['source', 'prospect_id', 'rule', 'severity', 'detail']

def validate_pipeline_records(pipeline_standardized, scenario_date):
    # TODO: Build issue records for each of the five validation rules.
    # Hint: use duplicated(), isna(), date comparisons, and str.fullmatch().
    # Keep one duplicate_prospect_id issue per duplicated prospect ID.
    raise NotImplementedError('Complete Task 2 before running this cell.')


In [ ]:
pipeline_issues = validate_pipeline_records(pipeline_standardized, SCENARIO_DATE)
expected_rules = {'duplicate_prospect_id', 'missing_stage_date', 'future_stage_date', 'unmapped_stage', 'invalid_zip'}
assert expected_rules.issubset(set(pipeline_issues['rule']))
assert len(pipeline_issues) == 5
display(pipeline_issues.sort_values(['severity', 'rule', 'prospect_id']))
print('Task 2 checks passed.')


## Task 3: Reconcile activity and campaign records

The CRM is the source of record for pipeline stage, but activity and campaign records can expose a bad operating picture. Flag four rules:

- activity_without_crm_record (HIGH): activity refers to a prospect absent from CRM.
- activity_ahead_of_crm_stage (HIGH): appointment activity shows a later stage than CRM.
- campaign_without_crm_record (MEDIUM): campaign record has no CRM prospect.
- source_conflict (MEDIUM): campaign and CRM records disagree on source code.

Use stage ranks of 3 for appointment_scheduled and 4 for appointment_completed. Retain only one CRM row per prospect when reconciling the duplicate P-1006 record.


In [ ]:
def reconcile_operational_sources(pipeline_standardized, activity, campaign):
    # TODO: Build the four reconciliation issue types described above.
    # Hint: create a one-row-per-prospect CRM view before merging source data.
    raise NotImplementedError('Complete Task 3 before running this cell.')


In [ ]:
reconciliation_issues = reconcile_operational_sources(pipeline_standardized, activity, campaign)
expected_rules = {'activity_without_crm_record', 'activity_ahead_of_crm_stage', 'campaign_without_crm_record', 'source_conflict'}
assert expected_rules.issubset(set(reconciliation_issues['rule']))
assert set(reconciliation_issues['prospect_id']) >= {'P-1002', 'P-1004', 'P-1013', 'P-1099'}
display(reconciliation_issues.sort_values(['severity', 'rule', 'prospect_id']))
print('Task 3 checks passed.')


## Task 4: Check label agreement before using labels

The pipeline integrity slides emphasize that a model cannot learn a reliable outcome from an unclear label. Two fictional analysts assigned a reason to each stalled-pipeline case. Compute the simple agreement rate, then inspect disagreements.

Interpret: Does disagreement mean a person made an error, or could it mean the rubric or evidence is underspecified?


In [ ]:
def simple_agreement_rate(labels):
    # TODO: Return the fraction of rows where analyst_a_label equals analyst_b_label.
    raise NotImplementedError('Complete Task 4 before running this cell.')


In [ ]:
agreement = simple_agreement_rate(labels)
assert agreement == 0.75
display(labels.loc[labels['analyst_a_label'].ne(labels['analyst_b_label'])])
print(f'Simple agreement rate: {agreement:.0%}')


## Produce the decision-readiness handoff

Run this cell after all four tasks pass. High-severity records are marked HOLD_FOR_REVIEW; they are not silently deleted. The report gives the next lab a bounded and traceable input.


In [ ]:
all_issues = pd.concat([pipeline_issues, reconciliation_issues], ignore_index=True)
high_severity_ids = set(all_issues.loc[all_issues['severity'].eq('HIGH'), 'prospect_id'])
handoff = pipeline_standardized.copy()
handoff['decision_status'] = handoff['prospect_id'].map(lambda prospect_id: 'HOLD_FOR_REVIEW' if prospect_id in high_severity_ids else 'ELIGIBLE_FOR_AGGREGATE_ANALYSIS')

issue_summary = all_issues.groupby(['source', 'severity', 'rule'], as_index=False).size().rename(columns={'size': 'issue_count'})
capacity_status = capacity.assign(status=capacity['as_of_date'].eq(SCENARIO_DATE).map({True: 'CURRENT', False: 'STALE'}))[['recruiter_id', 'as_of_date', 'status', 'capacity_note']]

handoff.to_csv(ARTIFACT_DIR / 'pipeline_handoff.csv', index=False)
issue_summary.to_csv(ARTIFACT_DIR / 'pipeline_issue_summary.csv', index=False)
capacity_status.to_csv(ARTIFACT_DIR / 'capacity_freshness.csv', index=False)
display(issue_summary)
display(capacity_status)
display(handoff[['prospect_id', 'canonical_stage', 'decision_status']])
print(f'Wrote handoff artifacts to: {ARTIFACT_DIR}')


## Debrief

1. Which issue would most distort a market-level recommender if left unresolved?
2. Which issue can a data engineer fix, and which requires the source owner or a recruiting leader?
3. What additional evidence would you require before treating a capacity snapshot as planning-ready?

### Optional extension

Add a source-freshness rule for campaign records. Then revise the label rubric so the two analysts can resolve ambiguous cases without forcing false consensus.


<!-- usard-next-colab-link -->
## Continue to Lab 2

[Open the next notebook in Colab](https://colab.research.google.com/github/TheDeafOne/USARD/blob/main/notebooks/02_recommender.ipynb).
